In [7]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
import nltk
nltk.download('stopwords')
nltk.download('wordnet')    
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [8]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
26978,"When it comes to movies, I don't easily discri...",positive
46036,The very first image of the movie shows a moun...,positive
39134,'Anne Christie' was Garbo's 14th film and the ...,positive
20398,This movie is definitely a case of style over ...,positive
45969,In 1990 Brad Pitt and Juiliette Lewis did a TV...,positive


In [9]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [10]:
df = normalize_text(df)
df.head()

,review,sentiment
26978,come movie easily discriminate crap pure crap ...,positive
46036,first image movie show mountain ridge early mo...,positive
39134,anne christie garbo s th film first husky swed...,positive
20398,movie definitely case style substance style go...,positive
45969,brad pitt juiliette lewis tv young die played ...,positive


In [11]:
df['sentiment'].value_counts()

sentiment
positive    255
negative    245
Name: count, dtype: int64

In [12]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [13]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
26978,come movie easily discriminate crap pure crap ...,1
46036,first image movie show mountain ridge early mo...,1
39134,anne christie garbo s th film first husky swed...,1
20398,movie definitely case style substance style go...,1
45969,brad pitt juiliette lewis tv young die played ...,1


In [14]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [15]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/anishgauri00786/IMDB-Sentiment-Analysis-Using-MLOps-principles.mlflow')
dagshub.init(repo_owner='anishgauri00786', repo_name='IMDB-Sentiment-Analysis-Using-MLOps-principles', mlflow=True)
mlflow.set_experiment("Baseline experiment")


Initialized MLflow to track repo "anishgauri00786/IMDB-Sentiment-Analysis-Using-MLOps-principles"

Repository anishgauri00786/IMDB-Sentiment-Analysis-Using-MLOps-principles initialized!

<Experiment: artifact_location='mlflow-artifacts:/50037e911e6f423eb515a8c8277eb70d', creation_time=1767177782589, experiment_id='0', last_update_time=1767177782589, lifecycle_stage='active', name='Baseline experiment', tags={}>

In [20]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2025-12-31 16:14:52,521 - INFO - Starting MLflow run...


2025-12-31 16:14:53,227 - INFO - Logging preprocessing parameters...
2025-12-31 16:14:54,418 - INFO - Initializing Logistic Regression model...
2025-12-31 16:14:54,420 - INFO - Fitting the model...
2025-12-31 16:14:54,476 - INFO - Model training complete.
2025-12-31 16:14:54,478 - INFO - Logging model parameters...
2025-12-31 16:14:54,859 - INFO - Making predictions...
2025-12-31 16:14:54,864 - INFO - Calculating evaluation metrics...
2025-12-31 16:14:54,903 - INFO - Logging evaluation metrics...
2025-12-31 16:14:56,818 - INFO - Saving and logging the model...
2025/12/31 16:14:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025-12-31 16:15:13,866 - INFO - Model training and logging completed in 20.64 seconds.
2025-12-31 16:15:13,868 - INFO - Accuracy: 0.648
2025-12-31 16:15:13,870 - INFO - Precision: 0.6111111111111112
2025-12-31 16:15:13,871 - INFO - Recall: 0.7333333333333333
2025-12-31 16:15:13,873 - INFO - F1 Score: 0.6666666666666666


🏃 View run rebellious-moose-550 at: https://dagshub.com/anishgauri00786/IMDB-Sentiment-Analysis-Using-MLOps-principles.mlflow/#/experiments/0/runs/d25e33b6c1f247d392446b9a4df093b3
🧪 View experiment at: https://dagshub.com/anishgauri00786/IMDB-Sentiment-Analysis-Using-MLOps-principles.mlflow/#/experiments/0
